# Analise Exploratoria - Risco de Inadimplencia de Credito (Brasil)

Dados reais do Banco Central do Brasil (SGS), de marco/2011 a maio/2026:
inadimplencia da carteira de credito (total, PF, PJ), Selic, IPCA,
desemprego e saldo da carteira de credito.


In [ ]:
import sys
sys.path.append("..")
import matplotlib.pyplot as plt
import seaborn as sns
from src.processamento import montar_painel_wide, montar_painel_long

sns.set_theme(style="whitegrid")
painel = montar_painel_wide()
painel.shape

In [ ]:
painel.head()

## 1. Trajetoria historica da inadimplencia por segmento

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for col, label in [("inad_total", "Total"), ("inad_pf", "Pessoa Fisica"), ("inad_pj", "Pessoa Juridica")]:
    ax.plot(painel["data"], painel[col], label=label)
ax.set_title("Taxa de inadimplencia da carteira de credito (%)")
ax.legend()
plt.show()

## 2. Selic e inadimplencia PF - existe relacao com defasagem?

A expectativa e que juros altos elevem a inadimplencia com alguns meses de atraso (o efeito nao e imediato).

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(painel["data"], painel["inad_pf"], color="tab:red", label="Inadimplencia PF")
ax1.set_ylabel("Inadimplencia PF (%)", color="tab:red")
ax2 = ax1.twinx()
ax2.plot(painel["data"], painel["selic_mensal"] * 12, color="tab:blue", alpha=0.6, label="Selic anualizada aprox.")
ax2.set_ylabel("Selic mensal x12 (%)", color="tab:blue")
plt.title("Inadimplencia PF vs. Selic")
plt.show()

## 3. Desemprego vs. inadimplencia

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=painel, x="desemprego", y="inad_pf", ax=ax)
ax.set_title("Desemprego x Inadimplencia PF (todos os meses)")
plt.show()

## 4. Correlacao entre as variaveis macro e a inadimplencia

In [ ]:
cols = ["inad_total", "inad_pf", "inad_pj", "selic_mensal", "ipca_mensal", "desemprego", "saldo_credito"]
corr = painel[cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matriz de correlacao")
plt.show()

## 5. O alvo do modelo: a inadimplencia sobe no mes seguinte?

In [ ]:
painel_long = montar_painel_long()
taxa_sobe = painel_long.groupby("segmento")["risco_subida"].mean()
print(taxa_sobe)
taxa_sobe.plot(kind="bar", title="Proporcao de meses em que a inadimplencia SOBE (por segmento)")
plt.ylabel("Proporcao")
plt.show()

## Conclusoes

- A inadimplencia de **pessoa fisica** e estruturalmente mais alta e mais
  volatil que a de pessoa juridica - faz sentido incluir `segmento` como
  feature categorica em vez de treinar um modelo unico ingenuo.
- Ha uma relacao visivel (com defasagem) entre Selic/desemprego em alta e
  inadimplencia em alta - por isso os lags e o acumulado de 3/12 meses
  entram como features em `src/processamento.py`.
- O alvo (`risco_subida`) fica proximo de 50/50 em todos os segmentos, ou
  seja, nao ha desbalanceamento severo de classes - bom sinal para o
  ROC-AUC ser uma metrica informativa sem precisar de tecnicas especiais
  de balanceamento alem do `class_weight="balanced"` ja usado no treino.
- O periodo 2011-2026 cobre cenarios bem diferentes (Selic a 2% em 2020,
  Selic de volta a patamares de dois digitos depois) - por isso o split
  cronologico (treino no passado, teste no periodo mais recente) e
  fundamental para uma avaliacao honesta do modelo.
